# Reddit keyword mention collection

This notebook scans archived subreddit dumps for keyword mentions and exports a CSV for downstream analysis.

In [ ]:
import json
import os
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import zstandard

pd.set_option('display.max_columns', 20)


In [ ]:
subreddits = ['addiction', 'fentanyl', 'Drugs', 'FentanylRecovery', 'opiates', 'OpiatesRecovery']
input_dir = Path('data/raw')
start_time = datetime(2021, 1, 1, tzinfo=timezone.utc).timestamp()

keyword_terms = [
    'naloxone',
    'narcan',
]
escaped_keywords = sorted((re.escape(term) for term in keyword_terms), key=len, reverse=True)
keyword_pattern = re.compile(r"\b(?:" + "|".join(escaped_keywords) + r")\b", re.IGNORECASE)

analysis_timezone = timezone.utc

input_dir.exists()


In [ ]:
def read_and_decode(reader, chunk_size, max_window_size, previous_chunk=None, bytes_read=0):
    chunk = reader.read(chunk_size)
    bytes_read += chunk_size
    if previous_chunk is not None:
        chunk = previous_chunk + chunk
    try:
        return chunk.decode()
    except UnicodeDecodeError:
        if bytes_read > max_window_size:
            raise UnicodeError(f'Unable to decode frame after reading {bytes_read:,} bytes')
        return read_and_decode(reader, chunk_size, max_window_size, chunk, bytes_read)


def read_lines_zst(file_name):
    with open(file_name, 'rb') as file_handle:
        buffer = ''
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(file_handle)
        while True:
            chunk = read_and_decode(reader, 2**27, (2**29) * 2)
            if not chunk:
                break
            lines = (buffer + chunk).split('\n')
            for line in lines[:-1]:
                yield line, file_handle.tell()
            buffer = lines[-1]
        reader.close()

def extract_text(obj, kind):
    if kind == 'submissions':
        # Keep submission text as selftext only.
        # Title is stored separately and can be modeled as another field.
        selftext = obj.get('selftext') or ''
        if selftext in {'[removed]', '[deleted]'}:
            selftext = ''
        text = selftext
    else:
        body = obj.get('body') or ''
        if body in {'[removed]', '[deleted]'}:
            body = ''
        text = body
    return text

def clean_text(text):
    return re.sub(r'\s+', ' ', text or '').strip()

def month_bucket(ts):
    return datetime.fromtimestamp(ts, tz=analysis_timezone).strftime('%Y-%m')



In [ ]:
mention_records = []

for subreddit in subreddits:
    for kind in ('submissions', 'comments'):
        file_path = input_dir / f'{subreddit}_{kind}.zst'
        if not file_path.exists():
            print(f'Missing file: {file_path}')
            continue
        for line, _ in tqdm(read_lines_zst(str(file_path)), desc=f'{subreddit}-{kind}', unit='rows'):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            created_utc = obj.get('created_utc')
            if created_utc is None:
                continue
            created_ts = int(created_utc)
            if created_ts < start_time:
                continue

            text = extract_text(obj, kind)
            if not text:
                continue

            matches = keyword_pattern.findall(text)
            if not matches:
                continue

            record = {
                'subreddit': subreddit,
                'kind': kind,
                'created_utc': created_ts,
                'created_dt': datetime.fromtimestamp(created_ts, tz=analysis_timezone),
                'month': month_bucket(created_ts),
                'mention_count': len(matches),
                'keyword_hits': sorted({m.lower() for m in matches}),
                'score': obj.get('score'),
                'author': obj.get('author'),
                'id': obj.get('id'),
                'text': clean_text(text),
            }
            if kind == 'submissions':
                record['title'] = obj.get('title') or ''
                record['num_comments'] = obj.get('num_comments')
                record['permalink'] = obj.get('permalink')
            else:
                record['parent_id'] = obj.get('parent_id')
                record['link_id'] = obj.get('link_id')
            mention_records.append(record)

print(f'Total naloxone mentions collected: {len(mention_records):,}')


In [ ]:
if mention_records:
    mentions_df = pd.DataFrame(mention_records)
    mentions_df['created_dt'] = pd.to_datetime(mentions_df['created_dt'])
else:
    mentions_df = pd.DataFrame(
        columns=['subreddit', 'kind', 'created_utc', 'created_dt', 'month', 'mention_count', 'keyword_hits', 'score', 'author', 'id', 'text']
    )

# 保存到文件
output_file = 'reddit_keyword_mentions.csv'
mentions_df.to_csv(output_file, index=False)
print(f'Saved {len(mentions_df):,} mentions to {output_file}')

# 显示前几条
display(mentions_df.head())


In [ ]:
# 在notebook中运行
print("数据质量分析：")
print(f"总数: {len(mentions_df):,}")
print(f"\n按长度分布:")
print(mentions_df['text'].str.len().describe())
print(f"\n按分数分布:")
print(mentions_df['score'].describe())
print(f"\n按subreddit分布:")
print(mentions_df['subreddit'].value_counts())